# ReFactX Knowledge-Graph Generation

This notebook uses the rule-backed API in `api.py` with ReFactX's callback-backed `KnowledgeGraphGeneration`. It does not use Postgres.

In [11]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path

# api.py uses data paths relative to the ReFactX-with-Rules directory.
RULES_DIR = Path.cwd()
if not (RULES_DIR / 'api.py').exists():
    RULES_DIR = (RULES_DIR / 'notebooks' / 'ReFactX-with-Rules').resolve()
os.chdir(RULES_DIR)
sys.path.insert(0, str(RULES_DIR))

import torch
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoProcessor, AutoModelForImageTextToText, TextStreamer
from transformers import ProcessorMixin

import refactx
from refactx.index import DictIndex
from refactx.generate import ConstrainedLogitsProcessor, ConstrainedStateList, KnowledgeGraphGeneration

from api import entity_index, property_strings, get
import api
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
MODEL = 'Qwen/Qwen3.5-4B'
NUM_BEAMS = 1
NUM_BATCHES = 1
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [13]:
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto', dtype='bfloat16')
tokenizer = processor
streamer = TextStreamer(tokenizer.tokenizer, skip_prompt=True)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

## Build the KG index and callbacks from `api.py`

`entity_index()` supplies the initial entities. `property_strings(entity)` supplies the available relations, and `get(subject, relation)` supplies the objects for each relation.

In [21]:
def _encode_index_text(text):
    tok = tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer
    return tok.encode(text, add_special_tokens=False)

index = DictIndex()
index.set_tokenizer(tokenizer)
entities = entity_index()
for entity in entities:
    # the initial is important because we will generate
    # something like <kg> <Subject> <pred> <obj>
    index.add(_encode_index_text(f' <{entity}>'))

# These are the callback signatures expected by KnowledgeGraphGeneration.
get_relations = property_strings
get_objects = get

print(f'Loaded {len(entities)} entities into the KG index.')

Loaded 100000 entities into the KG index.


## Register `KnowledgeGraphGeneration`

The `<kg>` marker activates graph traversal. Each generated path is serialized as `<entity> <relation> <object> .`. `long_chains=True` allows the generated object to become the next subject.

In [22]:
states = ConstrainedStateList(
    'auto',
    num_beams=NUM_BEAMS,
    num_batches=NUM_BATCHES,
    debug_tokenizer=tokenizer,
)

logits_processor = ConstrainedLogitsProcessor(states=states, tokenizer=tokenizer)
logits_processor.add_pattern(
    '<kg>',
    KnowledgeGraphGeneration,
    index=index,
    get_relations=get_relations,
    get_objects=get_objects,
    long_chains=False,
    eot=' </kg>\n',
)

In [29]:
prompt_messages = refactx.load_prompt('../../prompts/prompt_qwen36_angular2_kg_nothink.yaml')

In [30]:
def _tokenize(text):
    if isinstance(tokenizer, ProcessorMixin):
        return tokenizer.tokenizer(text, return_tensors='pt')
    return tokenizer(text, return_tensors='pt')

def _decode(ids):
    tok = tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer
    return tok.decode(ids, skip_special_tokens=True)

def ask(question, max_new_tokens=512, thinking=False, **generation_kwargs):
    logits_processor.reset_states()
    prompt = refactx.apply_prompt_template(
        tokenizer, question=question, enable_thinking=thinking, prompt_template=prompt_messages)
    inputs = _tokenize(prompt).to(model.device)
    model.eval()
    started = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            logits_processor=[logits_processor],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.80,
            top_k=20,
            min_p=0.0,
            repetition_penalty=1.0,
            num_beams=NUM_BEAMS,
            num_return_sequences=1,
            use_cache=True,
            streamer=streamer,
            **generation_kwargs,
        )
    state = logits_processor.states[0, 0]
    answer = _decode(output[0][inputs.input_ids.shape[1]:])
    paths = []
    for generation in state.generation_history:
        #if hasattr(generation, 'generated_path_metadata'):
        paths.extend(generation.generated_path_metadata)
    print(f'\n--- KG paths ({len(paths)}) ---')
    for path in paths:
        print(path)
    print(f'--- Elapsed: {time.time() - started:.2f}s ---')
    return answer, paths

## Ask questions

Run the following after the setup cells. The prompt should instruct the model to use `<kg>` for graph lookups.

In [31]:
qa = pd.read_csv(api.QA, keep_default_na=False)

In [32]:
qa['question'][0]

'Is it true that Priuli, Gremese e Guarato Group owns Babbo-Toselli SPA?'

In [34]:
qid = 0
print('Question', qa['question'][qid])
answer, paths = ask(qa['question'][qid])
print(answer)

Question Is it true that Priuli, Gremese e Guarato Group owns Babbo-Toselli SPA?
Bonanno<|im_start|>system
# Knowledge-Graph Question Answering

## Role
You are a question-answering system that answers using only verified triples retrieved from a knowledge graph.

## Core Principle
Ground every answer in triples returned by the knowledge graph. Do not use prior or parametric knowledge.

## Reasoning
Reason step-by-step in plain text. When information is missing, use a `<kg>` call and then continue reasoning from the returned triples. You may interleave reasoning and multiple `<kg>` calls.

## Knowledge-Graph Retrieval
Use `<kg>` to request graph information. ReFactX returns one or more triples enclosed by `<kg>` and `</kg>`:

<kg> <Subject> <relation> <Object> . </kg>

Examples of the triple format:

<kg> <Gremese e figli> <is_ultimate_control> <Gianni Sgalambro> . </kg>

<kg> <Gremese, Gargallo e Gagliardi SPA> <is_ultimate_control> <Valentina Travaglia-Burcardo> . </kg>

<kg> <Gremes

KeyboardInterrupt: 